# 17 - CUV ObjectCheck: wall labels (YOLOE), SAM3 gap fill, fine-tune queue

Adds the 2026-09-22 CUV_ObjectCheck collection to the detector.

1. **Wall labels** - YOLOE-26 (an open-vocabulary YOLO that takes text
   prompts) boxes the staged wall. Our detector never learned walls.
2. **SAM3 gap fill** - the SAM3 video tracker, seeded from each trace's
   labeled frames, fills the gaps the detector leaves (mostly the near range).
3. **Shared prep** - labels merged, traces split (per object: 2 test traces,
   one per speed, and 1 val trace), and fixed evaluation sets built: all of
   dataset_v2 val (old camera) and the CUV val traces. Every run and the
   baseline are scored on the same sets.
4. **Run queue** - each name in `RUNS` (recipes in the config's
   `experiments`) gets its own dataset (named by its replay share), trains,
   and is evaluated, back to back.
5. **Compare** - all runs, the first two runs, and the paper model side by side.

**Back-to-back safety.** Dataset and model folder names come from the
recipe, and the notebook refuses to reuse a folder that holds a different
recipe - add a new experiment name instead of editing one that has run. The
plan table in the config cell shows every run's folders and state before
anything long starts.

**Resume.** Stages 1-2 skip finished traces; a dataset materializes only
missing images; an interrupted training resumes from `last.pt`; finished
evaluations are skipped. After a disconnect, rerun from the top.

Inputs: `cuv_objectcheck/cuv_bundle_v1.zip` on Drive (built locally by
`scripts/export_cuv_bundle.py`), the dataset_v2 manifest, the paper model.
Config: `configs/experiments/cuv_finetune_v1.yaml`. Findings behind the
choices: `docs/cuv_objectcheck_eval.md`.

**GPU runtime required** (L4 or A100), and Hugging Face access to
`facebook/sam3` as for notebook 02.

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    assert (REPO_DIR / ".git").exists(), \
        "repo not found - run scripts/bootstrap_colab.sh first (set REPO_URL)"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None   # set to redirect all artifact I/O (e.g. for tests)
RAW_ROOT_OVERRIDE = None    # set to point at a different raw-sequence root

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")

In [ ]:
import json as _json
import shutil
import zipfile

import numpy as np
import pandas as pd
import torch
import yaml

from vision_uss_research.cuv_finetune import (BOX, DET_BIN, accept_sam3_segment,
                                              best_epoch, claim_run_dir,
                                              clean_track, cuv_manifest_rows,
                                              data_config, dataset_id,
                                              experiment_config, pick_box,
                                              plan_sam3_segments, replay_rows,
                                              spec_eval, split_traces,
                                              training_state)
from vision_uss_research.datasets.yolo_export import materialize_yolo_dataset
from vision_uss_research.runs import finish_run, new_run_id, start_run

cfg = yaml.safe_load((REPO_DIR / "configs" / "experiments" / "cuv_finetune_v1.yaml").read_text())
RUNS = ["yolo11s_ft_v2", "yolo11s_ft_ct"]   # auto vs click-track labels, ~1.5 h each; "yolo26s_full" ~8-10 h
RUN_SAM3 = True               # False skips stage 2 (e.g. no SAM3 access yet)
PREP_ONLY = False             # True on a CPU runtime: build the datasets, cache them on
                              # Drive as zips, stop; then switch to GPU and rerun from the top
SMOKE = False                 # True: two objects, 1 epoch, small images - pipeline check only
SMOKE_CLASSES = ["box30", "wall"]

DEVICE = 0 if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu")
# labels (stages 1-2) are shared by every run and by smoke runs
LABEL_DIR = ARTIFACTS / "labels" / "cuv" / f"labels_{cfg['tag']}"
if SMOKE:
    cfg["tag"] += "_smoke"    # smoke datasets/models never touch real folders
    cfg["frame_stride"] = 15
    cfg["train_defaults"].update(imgsz=320, batch=8)
    for e in cfg["experiments"].values():
        e["epochs"] = 1

BUNDLE_ZIP = ARTIFACTS / cfg["bundle"]
WORK = Path("/content/cuv") if IN_COLAB else ARTIFACTS / "cuv_work"
LOCAL_ROOT = Path("/content/dataset") if IN_COLAB else ARTIFACTS / "datasets_local"
EVAL_LOCAL = LOCAL_ROOT / f"eval_{cfg['tag']}"
BASELINE_DIR = ARTIFACTS / "models" / f"eval_{cfg['tag']}_baseline"
CACHE_DIR = ARTIFACTS / "datasets_cache"   # one zip per materialized dataset


def cache_restore(local, name):
    """Unpack a dataset zip cached by a CPU prep run (one large Drive read
    instead of thousands of small ones). Materialization then only fills gaps."""
    z = CACHE_DIR / f"{name}.zip"
    if (local / "dataset.yaml").exists() or not z.exists():
        return
    tmp = WORK / z.name
    print(f"restoring {name} from the Drive cache ...")
    shutil.copyfile(z, tmp)
    with zipfile.ZipFile(tmp) as f:
        f.extractall(local.parent)
    tmp.unlink()


def cache_save(local, name):
    """Zip a materialized dataset (stored, no compression - JPEGs) to Drive."""
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    tmp = WORK / f"{name}.zip"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_STORED) as f:
        for p in local.rglob("*"):
            if p.is_file():
                f.write(p, p.relative_to(local.parent))
    shutil.copyfile(tmp, CACHE_DIR / f"{name}.zip")
    tmp.unlink()
    print(f"cached {name} -> {CACHE_DIR / (name + '.zip')}")
for d in (WORK, LABEL_DIR):
    d.mkdir(parents=True, exist_ok=True)


def model_dir(name):
    return ARTIFACTS / "models" / new_run_id(cfg["stage"], f"{cfg['tag']}_{name}")


# fail fast: every queued run's folders are free or hold exactly its recipe
unknown = [n for n in RUNS if n not in cfg["experiments"]]
assert not unknown, f"not in cfg['experiments']: {unknown}"
plan = []
for name in RUNS:
    ec = experiment_config(cfg, name)
    did = dataset_id(cfg, ec["replay_frac"], ec.get("labels", "auto"))
    plan.append({
        "run": name, "init": ec["init"], "epochs": ec["epochs"],
        "optimizer": ec["optimizer"], "lr0": ec["lr0"], "replay": ec["replay_frac"],
        "dataset": did,
        "labels": ec.get("labels", "auto"),
        "dataset folder": claim_run_dir(ARTIFACTS / "datasets" / did,
                                        data_config(cfg, ec["replay_frac"], ec.get("labels", "auto"))),
        "model folder": claim_run_dir(model_dir(name), {**ec, "dataset": did}),
        "training": training_state(model_dir(name) / "train")})
print(f"device {DEVICE} | SAM3 {RUN_SAM3} | smoke {SMOKE} | labels {LABEL_DIR}")
display(pd.DataFrame(plan).set_index("run"))

## 0. Stage the bundle

In [ ]:
# one large Drive read beats 27k small ones: copy the zip, unzip on local disk
bundle_dir = WORK / Path(cfg["bundle"]).stem
if not IN_COLAB and BUNDLE_ZIP.with_suffix("").exists():
    bundle_dir = BUNDLE_ZIP.with_suffix("")          # local: already unpacked
if not (bundle_dir / "frames.csv").exists():
    assert BUNDLE_ZIP.exists(), (f"upload {BUNDLE_ZIP.name} to {BUNDLE_ZIP.parent} "
                                 "(scripts/export_cuv_bundle.py builds it)")
    local_zip = WORK / BUNDLE_ZIP.name
    if not local_zip.exists():
        shutil.copyfile(BUNDLE_ZIP, local_zip)
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(WORK)

frames = pd.read_csv(bundle_dir / "frames.csv")
if SMOKE:
    frames = frames[frames["label_class"].isin(SMOKE_CLASSES)]
IMG_H = 480


def frame_path(side, trace, f):
    return bundle_dir / "frames" / f"{side}_{trace}" / f"{int(f):05d}.jpg"


meta = (frames.groupby(["side", "trace"])
        .agg(label_class=("label_class", "first"), speed=("speed", "first"),
             height_bin=("height_bin", "first")).reset_index())
print(f"{len(frames)} frames within 8 m, {meta.shape[0]} traces")
display(frames.groupby("label_class")["label_src"]
        .agg(frames="size", labeled="count")
        .assign(coverage=lambda d: (d["labeled"] / d["frames"]).round(2)))

## 1. Wall labels with YOLOE

YOLOE is Ultralytics' open-vocabulary YOLO: instead of a fixed class list it
matches image regions against a text embedding, so `"wall"` works without
training. In the local probe (`scripts/wall_model_probe.py`) it found the
wall in 6/6 traces at 1.0-2.5 m and called nothing else a wall in 66
non-wall frames. It misses at 4 m (too small) and 0.5 m (the wall fills the
frame); stage 2 fills those. Boxes go through the same track cleanup as the
detector labels (spikes, gaps up to 20 frames interpolated).

In [ ]:
wall_csv = LABEL_DIR / "labels_yoloe_wall.csv"
if wall_csv.exists():
    wall_labels = pd.read_csv(wall_csv)
else:
    if IN_COLAB:  # text prompts need Ultralytics' CLIP fork
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/ultralytics/CLIP.git"], check=True)
    from ultralytics import YOLOE

    yc = cfg["yoloe"]
    yoloe = YOLOE(yc["weights"])
    yoloe.set_classes([yc["prompt"]])
    parts = []
    for (side, trace), g in frames[frames["label_class"] == "wall"].groupby(["side", "trace"]):
        g = g.sort_values("frame_idx")
        idx = g["frame_idx"].to_numpy()
        a_f, a_b = [], []
        for i in range(0, len(idx), 32):
            batch = [str(frame_path(side, trace, f)) for f in idx[i:i + 32]]
            for f, res in zip(idx[i:i + 32], yoloe.predict(
                    batch, imgsz=yc["imgsz"], conf=yc["conf"], device=DEVICE, verbose=False)):
                if len(res.boxes):  # one staged wall: the most confident box
                    k = int(res.boxes.conf.argmax())
                    a_f.append(int(f))
                    a_b.append(res.boxes.xyxy[k].cpu().numpy())
        tr = clean_track(np.array(a_f, int), np.array(a_b, float).reshape(-1, 4),
                         "wall", IMG_H, yc["max_gap"])
        parts.append(tr.assign(side=side, trace=trace,
                               src="yoloe_" + tr["src"]))
        print(f"{side}_{trace}: {len(a_f)} YOLOE boxes -> {len(tr)} labels")
    wall_labels = (pd.concat(parts, ignore_index=True) if parts else
                   pd.DataFrame(columns=["frame_idx", *BOX, "src", "side", "trace"]))
    wall_labels.to_csv(wall_csv, index=False)

w = wall_labels.merge(frames[["side", "trace", "frame_idx", "dist_m"]],
                      on=["side", "trace", "frame_idx"])
print(f"wall labels: {len(wall_labels)}")
display(w.groupby(pd.cut(w["dist_m"], [0, 0.5, 1, 1.6, 2.5, 4, 8]), observed=True).size()
        .rename("wall labels by distance (m)").to_frame())

In [ ]:
# look before trusting: 8 wall labels across distances
import cv2
import matplotlib.pyplot as plt


def show_labels(df, n=8, title=""):
    if df.empty:
        print("nothing to show")
        return
    pick = df.sort_values("dist_m").iloc[np.linspace(0, len(df) - 1, min(n, len(df))).astype(int)]
    fig, axes = plt.subplots(2, 4, figsize=(16, 6))
    for ax in axes.flat:
        ax.axis("off")
    for ax, r in zip(axes.flat, pick.itertuples()):
        img = cv2.cvtColor(cv2.imread(str(frame_path(r.side, r.trace, r.frame_idx))), cv2.COLOR_BGR2RGB)
        cv2.rectangle(img, (int(r.x0), int(r.y0)), (int(r.x1), int(r.y1)), (255, 200, 0), 3)
        ax.imshow(img)
        ax.set_title(f"{r.side} {r.trace} f{r.frame_idx} {r.dist_m:.1f} m {r.src}", fontsize=9)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


show_labels(w, title="YOLOE wall labels")

## 2. SAM3 gap fill

For every trace (metal bar excluded), labeled frames are the seeds. SAM3's
video tracker follows the object from a seed box through:

- **interior gaps** (5+ unlabeled frames between two labels): accepted only
  if the tracked mask still matches the label on the far side (IoU >= 0.3),
  otherwise the tracker drifted and the whole gap is dropped;
- **the tail** after the last label (up to 3 s - the near range the
  detector loses): accepted until the mask vanishes; curbs and boxes also
  stop when they leave through the bottom edge;
- **the lead** before the first label, for walls only (far frames YOLOE misses).

Per-trace results are saved, so a disconnect resumes where it stopped.

In [ ]:
sc = cfg["sam3"]
sam_dir = LABEL_DIR / "sam3"
sam_dir.mkdir(exist_ok=True)
excluded = {k for k, v in cfg["class_map"].items() if v is None}

base = frames[frames["label_src"].notna()][["side", "trace", "frame_idx", *BOX, "label_src"]]
seed_labels = pd.concat([base.rename(columns={"label_src": "src"}),
                         wall_labels[["side", "trace", "frame_idx", *BOX, "src"]]],
                        ignore_index=True)

if RUN_SAM3:
    from PIL import Image
    from vision_uss_research.labeling.sam3 import Sam3Engine

    engine = None   # loaded only if a trace still needs SAM3 (none after the first run)
    max_seg = sc["max_segment"]
    for (side, trace), g in frames.groupby(["side", "trace"]):
        label = g["label_class"].iloc[0]
        out_csv = sam_dir / f"{side}_{trace}.csv"
        if label in excluded or out_csv.exists():
            continue
        if engine is None:
            engine = Sam3Engine(score_threshold=sc["score_threshold"])
        lab = seed_labels[(seed_labels["side"] == side) & (seed_labels["trace"] == trace)]
        labeled = {int(r.frame_idx): np.array([r.x0, r.y0, r.x1, r.y1], float)
                   for r in lab.itertuples()}
        segs = plan_sam3_segments(sorted(g["frame_idx"]), labeled, sc["min_gap"],
                                  sc["max_tail"], backfill=label in sc["backfill_classes"],
                                  max_lead=sc["max_lead"])
        rows, skipped = [], 0
        for seg in segs:
            if len(seg["frames"]) > max_seg:   # tracker memory; too long to trust anyway
                skipped += 1
                continue
            seq = [seg["seed_frame"], *seg["frames"]]
            if seg["end_frame"] is not None:
                seq.append(seg["end_frame"])
            imgs = [Image.open(frame_path(side, trace, f)).convert("RGB") for f in seq]
            by_pos = engine.track(imgs, 0, {"box": tuple(map(float, seg["seed_box"]))})
            masks = {seq[i]: m for i, m in by_pos.items()}
            for f, b in accept_sam3_segment(seg, masks, label, IMG_H,
                                            sc["min_area_px"], sc["end_iou"]):
                rows.append({"side": side, "trace": trace, "frame_idx": f,
                             "x0": b[0], "y0": b[1], "x1": b[2], "y1": b[3], "src": "sam3"})
        pd.DataFrame(rows, columns=["side", "trace", "frame_idx", *BOX, "src"]).to_csv(out_csv, index=False)
        print(f"{side}_{trace} {label}: {len(segs)} segments ({skipped} too long) -> {len(rows)} frames")
    del engine

sam_files = sorted(sam_dir.glob("*.csv"))
sam_labels = (pd.concat([pd.read_csv(p) for p in sam_files], ignore_index=True)
              if sam_files else pd.DataFrame(columns=["side", "trace", "frame_idx", *BOX, "src"]))
print(f"SAM3 labels: {len(sam_labels)} frames from {len(sam_files)} traces")

In [ ]:
# look before trusting: SAM3 fills for a few objects
s = sam_labels.merge(frames[["side", "trace", "frame_idx", "dist_m", "label_class"]],
                     on=["side", "trace", "frame_idx"])
for label in ["box30", "curbhigh", "car", "wall"]:
    show_labels(s[s["label_class"] == label], title=f"SAM3 fills: {label}")

## 3. Shared prep: labels, split, fixed evaluation sets

Label priority per frame: detector/tracker (bundle) > YOLOE wall > SAM3.
Objects map into the existing classes (`class_map`): car, wall and boxes
become `high_other`, the adult dummy `dummychild`, both curbs `curbstone`;
metal bar is left out. Every 3rd labeled frame is kept (neighbours at 30 fps
are near-duplicates). The split depends only on the traces and the seed, so
every run tests on the same held-out traces.

In [ ]:
labels = pd.concat([seed_labels.assign(prio=0), sam_labels.assign(prio=2)],
                   ignore_index=True)
labels.loc[labels["src"].astype(str).str.startswith("yoloe"), "prio"] = 1
labels = (labels.sort_values("prio")
          .drop_duplicates(["side", "trace", "frame_idx"])
          .merge(meta[["side", "trace", "label_class"]], on=["side", "trace"]))

cov = frames.groupby("label_class").size().rename("frames").to_frame()
for col, p in (("detector_tracker", 0), ("yoloe", 1), ("sam3", 2)):
    cov[col] = labels[labels["prio"] == p].groupby("label_class").size()
cov = cov.fillna(0).astype(int)
cov["coverage"] = ((cov["detector_tracker"] + cov["yoloe"] + cov["sam3"]) / cov["frames"]).round(2)
display(cov)

# split from the FULL collection (as scripts/build_gold_review.py and notebooks
# 18-19 do), so the test traces never depend on which objects this run includes
full_meta = (pd.read_csv(bundle_dir / "frames.csv").groupby(["side", "trace"])
             .agg(label_class=("label_class", "first"), speed=("speed", "first")).reset_index())
split_map = split_traces(full_meta, cfg["split"]["n_test_per_group"],
                         cfg["split"]["n_val_per_group"], cfg["split"]["seed"])
meta["split"] = [split_map[(s, int(t))] for s, t in zip(meta["side"], meta["trace"])]
display(meta.groupby(["label_class", "split"]).size().unstack(fill_value=0))
cuv = cuv_manifest_rows(labels, split_map, cfg["class_map"], cfg["class_ids"],
                        stride=cfg["frame_stride"])
CUV_ROWS = {"auto": cuv}
if any(experiment_config(cfg, n).get("labels") == "clicktrack" for n in RUNS):
    # notebook 19 tracks replace the auto labels on the traces they cover
    # (the 48 train/val traces); test traces keep the auto labels, which only
    # fill the manifest's test split and are never trained on
    ct = pd.read_csv(ARTIFACTS / cfg["clicktrack_labels"])
    assert not set(zip(ct["side"], ct["trace"])) & set(
        (s, t) for (s, t), v in split_map.items() if v == "test"), "click-track file contains test traces"
    ct = (ct[ct["status"] == "box"][["side", "trace", "frame_idx", *BOX]]
          .merge(meta[["side", "trace", "label_class"]], on=["side", "trace"]))
    covered = set(zip(ct["side"], ct["trace"]))
    keep = labels[[(s, t) not in covered for s, t in zip(labels["side"], labels["trace"])]]
    CUV_ROWS["clicktrack"] = cuv_manifest_rows(
        pd.concat([keep, ct], ignore_index=True), split_map, cfg["class_map"],
        cfg["class_ids"], stride=cfg["frame_stride"])
    print(f"click-track labels: {len(ct)} frames on {len(covered)} traces -> "
          f"{len(CUV_ROWS['clicktrack'])} manifest rows (auto: {len(cuv)})")
class_names = [n for n, _ in sorted(cfg["class_ids"].items(), key=lambda kv: kv[1])]
excluded = {k for k, v in cfg["class_map"].items() if v is None}

base_dir = ARTIFACTS / cfg["base_dataset"]
if (base_dir / "dataset_manifest.csv").exists():
    prov = _json.loads((base_dir / "provenance.json").read_text())
    assert prov["class_ids"] == cfg["class_ids"], "class ids differ from dataset_v2"
    base_manifest = pd.read_csv(base_dir / "dataset_manifest.csv")
    # manifests built before target_object was added (dataset_v2 is one) lack it
    if "target_object" not in base_manifest.columns:
        id_to_name = {v: k for k, v in cfg["class_ids"].items()}
        base_manifest["target_object"] = base_manifest["class_id"].map(id_to_name)
    base_roots = {k: Path(v) for k, v in prov["frames_roots"].items()}
else:
    assert SMOKE or not IN_COLAB, f"dataset_v2 manifest not found at {base_dir}"
    print(f"WARNING: no dataset_v2 manifest at {base_dir} - no replay, no val_old")
    base_manifest, base_roots = None, {}
frames_roots = {"cuv_bundle": bundle_dir / "frames", **base_roots}

# fixed evaluation sets: ALL of dataset_v2 val (not a replay sample) + CUV val
eval_rows = cuv[cuv["split"] == "val"]
if base_manifest is not None:
    eval_rows = pd.concat([base_manifest[base_manifest["split"] == "val"][cuv.columns],
                           eval_rows], ignore_index=True)
EVAL_LOCAL.mkdir(parents=True, exist_ok=True)
eval_rows.to_csv(EVAL_LOCAL / "eval_manifest.csv", index=False)
import hashlib
EVAL_CACHE = f"{EVAL_LOCAL.name}_{hashlib.sha1((EVAL_LOCAL / 'eval_manifest.csv').read_bytes()).hexdigest()[:6]}"
cache_restore(EVAL_LOCAL, EVAL_CACHE)
materialize_yolo_dataset(EVAL_LOCAL / "eval_manifest.csv", frames_roots, EVAL_LOCAL,
                         class_names=class_names)
if PREP_ONLY and not (CACHE_DIR / f"{EVAL_CACHE}.zip").exists():
    cache_save(EVAL_LOCAL, EVAL_CACHE)
val_imgs = sorted((EVAL_LOCAL / "images" / "val").rglob("*.jpg"))
EVAL_SETS = {}
for vname, ps in {"val_old": [p for p in val_imgs if not p.parent.name.startswith("cuv_")],
                  "val_cuv": [p for p in val_imgs if p.parent.name.startswith("cuv_")]}.items():
    if not ps:
        continue
    (EVAL_LOCAL / f"{vname}.txt").write_text("\n".join(map(str, ps)) + "\n")
    y = EVAL_LOCAL / f"{vname}.yaml"
    y.write_text(yaml.safe_dump({"path": str(EVAL_LOCAL), "train": f"{vname}.txt",
                                 "val": f"{vname}.txt", "names": class_names}))
    EVAL_SETS[vname] = y
    print(f"{vname}: {len(ps)} images")

test = frames.merge(meta[meta["split"] == "test"][["side", "trace"]], on=["side", "trace"])
# label boxes (detector/tracker, YOLOE, SAM3) let the spec eval check WHERE the box is
test_labels = labels[["side", "trace", "frame_idx", *BOX]].rename(
    columns={c: "l" + c for c in BOX})
EVAL_KEY = _json.loads(_json.dumps({
    "metric_version": 3,
    "test_traces": sorted(f"{s}_{t}" for s, t in test[["side", "trace"]].drop_duplicates().values),
    "eval": cfg["eval"], "map_imgsz": cfg["train_defaults"]["imgsz"],
    "val_images": {k: len((EVAL_LOCAL / f"{k}.txt").read_text().split()) for k in EVAL_SETS}}))
print(f"test traces: {len(EVAL_KEY['test_traces'])}")

## 4. Run queue

Functions first, then the loop. Per run: build (or reuse) its dataset ->
train (fresh, resumed, or skipped when done) -> evaluate on the fixed sets.
The baseline and the first two runs are evaluated once on the same sets.

Evaluation per model:

- **mAP** on `val_old` (all of dataset_v2 val: did the old camera get
  worse?) and `val_cuv` (CUV val traces), at the training image size.
- **SOP spec** on the held-out CUV test traces at each `eval.imgsz`: the
  corridor rule picks one box per frame; a frame is a HIT when its Low/High
  bin is right; a trace passes when >= 50% of frames within 0.25 m of the
  spec distance are HITs. The **localized** pass also needs the box to
  overlap the frame's label box (IoU >= 0.3) - without it a model boxing
  everything as `high_other` would pass every high object.

In [ ]:
from ultralytics import YOLO


def build_dataset(ec):
    lab = ec.get("labels", "auto")
    cuv = CUV_ROWS[lab]
    did = dataset_id(cfg, ec["replay_frac"], lab)
    ddir, local = ARTIFACTS / "datasets" / did, LOCAL_ROOT / did
    dconf = data_config(cfg, ec["replay_frac"], lab)
    if claim_run_dir(ddir, dconf) == "new":
        old = (replay_rows(base_manifest, ec["replay_frac"], seed=cfg["split"]["seed"])
               if base_manifest is not None else pd.DataFrame(columns=cuv.columns))
        manifest = pd.concat([old[cuv.columns], cuv], ignore_index=True)
        start_run(ddir, config=dconf, repo_root=REPO_DIR,
                  inputs=[cfg["base_dataset"], cfg["bundle"], str(LABEL_DIR.relative_to(ARTIFACTS))])
        manifest.to_csv(ddir / "dataset_manifest.csv", index=False)
        meta.to_csv(ddir / "trace_split.csv", index=False)
        (ddir / "provenance.json").write_text(_json.dumps({
            "class_ids": cfg["class_ids"], "class_names": class_names,
            "frames_roots": {k: str(v) for k, v in frames_roots.items()},
            "rows_per_split_source": {f"{sp}/{src}": int(n) for (sp, src), n in
                                      manifest.groupby(["split", "source"]).size().items()}},
            indent=1))
    cache_restore(local, did)
    result = materialize_yolo_dataset(ddir / "dataset_manifest.csv", frames_roots, local,
                                      class_names=class_names,
                                      heartbeat_file=ddir / "progress.json")
    if PREP_ONLY and not (CACHE_DIR / f"{did}.zip").exists():
        cache_save(local, did)
    rows = pd.read_csv(ddir / "dataset_manifest.csv", usecols=["split", "source"])
    finish_run(ddir, summary={"rows": int(len(rows)),
                              "replay_rows": int((rows["source"] != "cuv_bundle").sum())})
    print(f"dataset {did}: {result['images_written']} written, "
          f"{result['images_skipped_existing']} already there, {len(result['missing_frames'])} missing")
    return did, local


def train(name, ec, did, local):
    mdir = model_dir(name)
    tdir = mdir / "train"
    if claim_run_dir(mdir, {**ec, "dataset": did}) == "new":
        start_run(mdir, config={**ec, "dataset": did},
                  inputs=[f"datasets/{did}"], repo_root=REPO_DIR)
    state = training_state(tdir)
    if state == "resume":
        print(f"{name}: resuming from last.pt")
        YOLO(str(tdir / "weights" / "last.pt")).train(resume=True)
    elif state == "fresh":
        init = str(ARTIFACTS / cfg["base_model"]) if ec["init"] == "base_model" else ec["init"]
        a = ec["augment"]
        YOLO(init).train(
            data=str(local / "dataset.yaml"), epochs=ec["epochs"], imgsz=ec["imgsz"],
            batch=ec["batch"], seed=ec["seed"], device=DEVICE,
            optimizer=ec["optimizer"], lr0=ec["lr0"], warmup_epochs=ec["warmup_epochs"],
            freeze=ec["freeze"], mosaic=a["mosaic"], copy_paste=a["copy_paste"],
            fliplr=a["fliplr"], scale=a["scale"],
            project=str(mdir), name="train", exist_ok=True)
    if state != "done":
        (tdir / "train_done.json").write_text(_json.dumps(
            {"best_epoch": best_epoch(tdir / "results.csv")}))
    print(f"{name}: trained, best.pt = epoch {best_epoch(tdir / 'results.csv')}")
    return mdir


def predict_bins(weights, imgsz):
    m = YOLO(str(weights))
    out = []
    for (side, trace), g in test.groupby(["side", "trace"]):
        idx = g.sort_values("frame_idx")["frame_idx"].to_numpy()
        for i in range(0, len(idx), 32):
            batch = [str(frame_path(side, trace, f)) for f in idx[i:i + 32]]
            for f, res in zip(idx[i:i + 32], m.predict(batch, imgsz=imgsz, conf=cfg["eval"]["conf"],
                                                       device=DEVICE, verbose=False)):
                b, c = res.boxes.xyxy.cpu().numpy(), res.boxes.conf.cpu().numpy()
                k = pick_box(b, c, res.orig_shape[1], res.orig_shape[0], cfg["eval"]["min_score"])
                px = b[k] if k is not None else [np.nan] * 4
                out.append({"side": side, "trace": trace, "frame_idx": int(f),
                            "det_bin": None if k is None else DET_BIN[m.names[int(res.boxes.cls[k])]],
                            "px0": px[0], "py0": px[1], "px1": px[2], "py1": px[3]})
    return (test.merge(pd.DataFrame(out), on=["side", "trace", "frame_idx"])
                .merge(test_labels, on=["side", "trace", "frame_idx"], how="left"))


def evaluate(weights, out_dir, label):
    edir = Path(out_dir) / "eval_fixed"
    done = edir / "eval_done.json"
    if done.exists() and _json.loads(done.read_text()).get("key") == EVAL_KEY:
        print(f"{label}: already evaluated")
        return
    edir.mkdir(parents=True, exist_ok=True)
    rows = []
    for vname, y in EVAL_SETS.items():
        m = YOLO(str(weights)).val(data=str(y), split="val", imgsz=EVAL_KEY["map_imgsz"],
                                   device=DEVICE, verbose=False, plots=False)
        rows.append({"val": vname, "mAP50": round(float(m.box.map50), 3),
                     "mAP50_95": round(float(m.box.map), 3)})
    pd.DataFrame(rows, columns=["val", "mAP50", "mAP50_95"]).to_csv(edir / "val_map.csv", index=False)
    per = [spec_eval(predict_bins(weights, sz), window=cfg["eval"]["spec_window_m"])[0].assign(imgsz=sz)
           for sz in cfg["eval"]["imgsz"]]
    pd.concat(per).to_csv(edir / "spec_per_trace.csv", index=False)
    done.write_text(_json.dumps({"key": EVAL_KEY, "label": label, "weights": str(weights)}))
    print(f"{label}: evaluated")

In [ ]:
if PREP_ONLY:
    for name in RUNS:
        print(f"\n===== {name} (prep only)")
        build_dataset(experiment_config(cfg, name))
    print("\nprep done - datasets cached on Drive. Switch to a GPU runtime "
          "(Runtime > Change runtime type), set PREP_ONLY = False, rerun from the top.")
else:
    evaluate(ARTIFACTS / cfg["base_model"], BASELINE_DIR, "baseline")
    for name in cfg.get("legacy_runs", []):
        w = ARTIFACTS / "models" / name / "train" / "weights" / "best.pt"
        if w.exists():   # smoke results go to their own folder, not the real run's
            out = ARTIFACTS / "models" / (f"eval_{cfg['tag']}_{name}" if SMOKE else name)
            evaluate(w, out, name)

    for name in RUNS:
        print(f"\n===== {name}")
        ec = experiment_config(cfg, name)
        did, local = build_dataset(ec)
        mdir = train(name, ec, did, local)
        evaluate(mdir / "train" / "weights" / "best.pt", mdir, name)
        _ = finish_run(mdir, summary={"dataset": did,
                                      "best_epoch": best_epoch(mdir / "train" / "results.csv")})

## 5. Compare

In [ ]:
size = cfg["eval"]["imgsz"][0]
bins = meta.drop_duplicates("label_class").set_index("label_class")["height_bin"]
entries = [("baseline (paper model)", BASELINE_DIR, None)]
for name in cfg.get("legacy_runs", []):
    d = ARTIFACTS / "models" / name
    entries.append((f"{name} (first run)",
                    ARTIFACTS / "models" / f"eval_{cfg['tag']}_{name}" if SMOKE else d, d / "train"))
entries += [(name, model_dir(name), model_dir(name) / "train") for name in RUNS]

summary, passes = [], {}
for label, d, tdir in entries:
    e = Path(d) / "eval_fixed"
    if not (e / "eval_done.json").exists():
        continue
    vm = pd.read_csv(e / "val_map.csv").set_index("val")
    sp = pd.read_csv(e / "spec_per_trace.csv")
    sp = sp[sp["imgsz"] == size].assign(bin=lambda x: x["label_class"].map(bins))
    trained = sp[~sp["label_class"].isin(excluded)]
    checked = trained[trained["passes_loc"].notna()]   # window had label boxes
    summary.append({
        "run": label,
        "best epoch": best_epoch(tdir / "results.csv") if tdir else None,
        "val_old mAP50": vm["mAP50"].get("val_old"),
        "val_old mAP50-95": vm["mAP50_95"].get("val_old"),
        "val_cuv mAP50": vm["mAP50"].get("val_cuv"),
        f"spec traces passing @{size}": f"{int(trained['passes'].sum())}/{len(trained)}",
        f"... with box on the object @{size}": (
            f"{int(checked['passes_loc'].astype(bool).sum())}/{len(checked)}"
            + (f" ({len(trained) - len(checked)} unchecked)" if len(checked) < len(trained) else "")),
        f"worst High called Low @{size}": sp.loc[sp["bin"] == "high", "wrong_bin_rate"].max(),
        f"worst Low called High @{size}": sp.loc[sp["bin"] == "low", "wrong_bin_rate"].max(),
        f"wooden curb hit @{size}": round(sp.loc[sp["label_class"] == "woodencurb10",
                                                 "window_hit_rate"].mean(), 2)})
    passes[label] = sp.groupby("label_class")["passes"].agg(lambda x: f"{int(x.sum())}/{len(x)}")

if not summary:
    raise SystemExit("nothing evaluated yet (prep-only run) - switch to GPU and rerun")
comparison = pd.DataFrame(summary).set_index("run")
display(comparison)
display(pd.DataFrame(passes))
comparison.to_csv(ARTIFACTS / "models" / f"compare_{cfg['tag']}.csv")

## Reading the comparison

Pick the run that, in this order:

1. has **0 for "worst High called Low"** - a high obstacle reported as
   drivable is the error that matters for safety;
2. keeps **val_old mAP50 within ~0.02 of the baseline** (no forgetting of
   the old camera; this is all of dataset_v2 val, so runs are comparable);
3. passes the most spec traces **with the box on the object** (a pass
   without it is not trustworthy), then has the higher wooden-curb hit rate.

**best epoch** near 1 means the recipe disturbed the starting weights (the
first `yolo11s_ft` run); near the last epoch means more epochs could help.
With 2 test traces per object (4 for pole and car), the spec columns show
direction, not certified rates. Next for the winner: notebook 07 on the
dataset_v2 gold test split, then ONNX export (deployment plan, phase P1).